# Jira Backlog Management Prototype
Fetches Jira issues, clusters them (TF-IDF + K-Means), detects duplicates via cosine similarity, and uses an LLM (Gemini) with RAG context to enhance cluster names, duplicate-pair insights, and an executive summary.

**Setup:** requires a `GEMINI_API_KEY` environment variable (see Section 1). This version runs in any local Jupyter environment, not just Google Colab.

## 1. Setup

In [ ]:
import sys
!{sys.executable} -m pip install -q langchain_google_genai python-dotenv

import os, re, json, time
from datetime import timezone

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from bs4 import BeautifulSoup

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

for pkg, resource in [("stopwords", "corpora/stopwords"),
                      ("punkt", "tokenizers/punkt"),
                      ("punkt_tab", "tokenizers/punkt_tab/english.pickle")]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(pkg)

# Reads from the GEMINI_API_KEY environment variable.
# Easiest setup: copy .env.example to .env in this folder and fill in your key.
# Alternatively, export it directly in your shell, e.g.:
#   export GEMINI_API_KEY="your-key-here"       (macOS/Linux)
#   $env:GEMINI_API_KEY = "your-key-here"        (Windows PowerShell)

try:
    from dotenv import load_dotenv
    load_dotenv()  # loads variables from a local .env file, if present
except ImportError:
    pass  # python-dotenv not installed; rely on the shell environment instead

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY environment variable not set. "
        "Copy .env.example to .env and add your key, or export it in your shell before running this notebook."
    )

llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3, api_key=API_KEY)

## 2. Fetch Jira Issues

In [ ]:
JIRA_API_URL = "https://issues.apache.org/jira/rest/api/2/search"
JIRA_PARAMS = {
    "jql": "",
    "maxResults": 500,
    "fields": "summary,description,status,created,updated,issuetype,priority,reporter,assignee,comment,labels",
}

def fetch_jira_issues(url, params):
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    issues = resp.json().get("issues", [])

    rows = []
    for issue in issues:
        f = issue.get("fields", {})
        rows.append({
            "id": issue.get("id"),
            "key": issue.get("key"),
            "summary": f.get("summary"),
            "description": f.get("description"),
            "status": (f.get("status") or {}).get("name"),
            "created": f.get("created"),
            "updated": f.get("updated"),
            "issuetype": (f.get("issuetype") or {}).get("name"),
            "priority": (f.get("priority") or {}).get("name"),
            "reporter": (f.get("reporter") or {}).get("displayName"),
            "assignee": (f.get("assignee") or {}).get("displayName"),
            "labels": f.get("labels"),
            "comments": [c.get("body") for c in f.get("comment", {}).get("comments", [])],
        })

    df = pd.DataFrame(rows)
    df["created"] = pd.to_datetime(df["created"])
    df["updated"] = pd.to_datetime(df["updated"])
    return df

df_jira_issues = fetch_jira_issues(JIRA_API_URL, JIRA_PARAMS)
print(f"Fetched {len(df_jira_issues)} issues.")
df_jira_issues.head()

## 3. Preprocess Text

In [ ]:
STOP_WORDS = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return []
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text).lower()
    tokens = word_tokenize(text)
    return [t for t in tokens if t not in STOP_WORDS]

df_jira_issues["full_text"] = df_jira_issues["summary"].fillna("") + " " + df_jira_issues["description"].fillna("")
df_jira_issues["processed_text"] = df_jira_issues["full_text"].apply(clean_text)
df_jira_issues[["summary", "processed_text"]].head()

## 4. TF-IDF Vectorization

In [ ]:
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, lowercase=False)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_jira_issues["processed_text"])
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

## 5. K-Means Clustering

In [ ]:
NUM_CLUSTERS = 5
NUM_TOP_KEYWORDS = 10

kmeans_model = KMeans(n_clusters=NUM_CLUSTERS, init="k-means++", max_iter=300, random_state=42, n_init="auto")
df_jira_issues["cluster"] = kmeans_model.fit_predict(tfidf_matrix)
cluster_counts = df_jira_issues["cluster"].value_counts().sort_index()
print(cluster_counts.to_string())

## 6. Top Keywords per Cluster

In [ ]:
def top_keywords_for_cluster(cluster_id, n=NUM_TOP_KEYWORDS):
    centroid = kmeans_model.cluster_centers_[cluster_id]
    top_idx = centroid.argsort()[:-n - 1:-1]
    return [feature_names[i] for i in top_idx]

cluster_keywords = {i: top_keywords_for_cluster(i) for i in range(NUM_CLUSTERS)}
for cid, kw in cluster_keywords.items():
    print(f"Cluster {cid}: {', '.join(kw)}")

## 7. Duplicate Detection (Cosine Similarity)

In [ ]:
SIMILARITY_THRESHOLD = 0.8
cosine_sim_matrix = cosine_similarity(tfidf_matrix)

duplicate_pairs = []
for i in range(cosine_sim_matrix.shape[0]):
    for j in range(i + 1, cosine_sim_matrix.shape[1]):
        sim = cosine_sim_matrix[i, j]
        if sim > SIMILARITY_THRESHOLD:
            duplicate_pairs.append({
                "issue1_key": df_jira_issues.loc[i, "key"],
                "issue1_summary": df_jira_issues.loc[i, "summary"],
                "issue2_key": df_jira_issues.loc[j, "key"],
                "issue2_summary": df_jira_issues.loc[j, "summary"],
                "similarity": sim,
            })

print(f"Found {len(duplicate_pairs)} potential duplicate pairs (threshold > {SIMILARITY_THRESHOLD}).")

## 8. RAG Context Files
Dummy knowledge-base files (release notes, project context, 2026 themes) used to ground the LLM.

In [ ]:
RAG_FILES = {
    "release_notes.txt": """
Release Notes - Version 3.8.1
Release Date: May 1, 2026
*   Bug Fix: Addressed critical security vulnerability (CVE-2026-XXXX).
*   Improvement: Enhanced performance of client connection handling.
*   Feature: Introduced new metrics for Prometheus integration.

Release Notes - Version 3.8.0
Release Date: April 15, 2026
*   Major Feature: Implemented SASL authentication for client connections.
*   Bug Fix: Resolved an issue with session expiration during network partitions.
*   Improvement: Updated dependency `netty-tcnative` to 2.0.60.Final.

Release Notes - Version 3.7.1
Release Date: March 10, 2026
*   Bug Fix: Fixed an NPE in `ConnectionMetricsTest`.
*   Improvement: Optimized leader election process for faster failover.
*   Feature: Added support for auto-reloading client key/trust stores.

Release Notes - Version 3.7.0
Release Date: February 20, 2026
*   Feature: Introduced Admin Server APIs for enhanced management.
*   Improvement: Refactored logging to use SLF4J and Logback.
*   Bug Fix: Corrected issue with data directory size reporting.

Release Notes - Version 3.6.3
Release Date: January 5, 2026
*   Security Fix: Addressed minor vulnerabilities in third-party libraries.
*   Improvement: Better handling of dynamic configuration file updates.
*   Feature: Expanded unit test coverage for consistency semantics.
""",
    "project_context.txt": """
Project Context: Jira Backlog Management Prototype

Goals:
*   Improve efficiency in Jira backlog management.
*   Automate identification of duplicate issues.
*   Facilitate intelligent clustering and categorization of issues.
*   Provide actionable insights for project managers and development teams.

Description:
This project aims to develop a prototype system for advanced Jira backlog management using natural language processing (NLP) and machine learning techniques. It involves fetching Jira issues, preprocessing their textual content, and applying clustering algorithms to group similar issues. Additionally, it identifies potential duplicate issues to streamline the backlog and reduce redundancy. The system also extracts key themes from clusters to offer high-level insights.

Process:
1.  Data Retrieval: Fetch up to 500 Jira issues via the Jira REST API.
2.  Text Preprocessing: Clean and tokenize issue summaries and descriptions.
3.  Feature Engineering: Convert text into numerical TF-IDF vectors.
4.  Clustering: Apply K-Means to group issues into 5 clusters.
5.  Keyword Extraction: Identify representative keywords for each cluster.
6.  Duplicate Detection: Calculate cosine similarity to find highly similar issues.
7.  Analysis & Reporting: Summarize findings and visualize distributions.

Workflow:
*   Initial Setup: Define API credentials and endpoint.
*   Execution: Run the Python script to perform data retrieval and analysis.
*   Review Results: Examine cluster assignments, keywords, and duplicate pairs.
*   Actionable Insights: Utilize the generated insights to refine Jira backlog, assign issues, or merge duplicates.
*   Iteration: Continuously refine models and parameters based on feedback and new data.
""",
    "2026_theme.txt": """
2026 Themes and Increment Purpose

Overall Increment Purpose:
To enhance the stability, scalability, and security of our core systems while introducing innovative features that improve user experience and operational efficiency. Focus will be on reducing technical debt, optimizing performance, and preparing for future growth.

FY26Q1 Themes (January - March 2026):
*   Theme 1: Core System Hardening: Focus on identifying and patching critical vulnerabilities, improving error handling, and implementing robust logging and monitoring solutions.
*   Theme 2: Performance Optimization: Initiatives to reduce latency, optimize database queries, and improve overall system responsiveness for high-traffic operations.
*   Theme 3: User Authentication Revamp: Upgrade authentication mechanisms to support modern standards and improve security posture.

FY26Q2 Themes (April - June 2026):
*   Theme 1: Scalability Enhancements: Develop and deploy solutions for dynamic resource allocation, load balancing, and horizontal scaling to support increased user loads.
*   Theme 2: Data Integrity & Governance: Implement stricter data validation rules, introduce data lineage tracking, and improve data backup and recovery processes.
*   Theme 3: API Modernization: Refactor and standardize existing APIs, and introduce new GraphQL endpoints for more flexible data access.

FY26Q3 Themes (July - September 2026):
*   Theme 1: Advanced Analytics Integration: Integrate with new analytics platforms to provide deeper insights into user behavior and system performance.
*   Theme 2: AI/ML Feature Development: Pilot and develop initial AI/ML-driven features, such as intelligent recommendations or automated anomaly detection.
*   Theme 3: Developer Experience Improvement: Streamline CI/CD pipelines, improve documentation, and develop internal tools to boost developer productivity.

FY26Q4 Themes (October - December 2026):
*   Theme 1: Cloud Migration Readiness: Assess and prepare core applications for potential migration to a cloud-native architecture, focusing on containerization and microservices.
*   Theme 2: Compliance & Regulatory Adherence: Ensure all systems and processes meet evolving industry compliance standards and regulatory requirements.
*   Theme 3: Strategic Partnership Integrations: Build and refine integrations with key third-party partners to expand ecosystem capabilities and value proposition.
""",
}

for filename, content in RAG_FILES.items():
    with open(filename, "w") as f:
        f.write(content)

release_notes_context = RAG_FILES["release_notes.txt"]
project_context = RAG_FILES["project_context.txt"]
theme_2026_context = RAG_FILES["2026_theme.txt"]
print("RAG context files written:", ", ".join(RAG_FILES))

## 9. LLM: Enhanced Cluster Names & Descriptions

In [ ]:
cluster_enhancement_prompt = PromptTemplate(
    input_variables=["release_notes_context", "project_context", "theme_2026_context", "cluster_keywords", "cluster_id"],
    template=(
        "You are an expert in Jira issue analysis and project management. Provide a comprehensive analysis "
        "for a Jira issue cluster: a descriptive name, its alignment with 2026 themes, a detailed description, "
        "and actionable Project Manager recommendations.\n\n"
        "Project context:\n{project_context}\n\n"
        "Past release notes:\n{release_notes_context}\n\n"
        "2026 Themes:\n{theme_2026_context}\n\n"
        "Top keywords for Cluster {cluster_id}:\n{cluster_keywords}\n\n"
        "Respond in JSON with keys: name (5-8 words), theme_alignment (2-3 sentences), "
        "description (3-5 sentences), pm_recommendations (3-4 sentences)."
    )
)

def call_llm_json(prompt_text):
    """Invoke the LLM and parse a JSON object out of the response (handles ```json fences)."""
    response = llm.invoke(prompt_text)
    content = response.content.strip()
    match = re.search(r"```json\s*(.*?)\s*```", content, re.DOTALL)
    return json.loads(match.group(1).strip() if match else content)

enhanced_cluster_details = {}
for cid in range(NUM_CLUSTERS):
    prompt = cluster_enhancement_prompt.format(
        release_notes_context=release_notes_context,
        project_context=project_context,
        theme_2026_context=theme_2026_context,
        cluster_keywords=", ".join(cluster_keywords[cid]),
        cluster_id=cid,
    )
    try:
        enhanced_cluster_details[cid] = call_llm_json(prompt)
    except json.JSONDecodeError:
        enhanced_cluster_details[cid] = {"name": "Error", "theme_alignment": "Error",
                                          "description": "Error", "pm_recommendations": "Error"}
    print(f"Cluster {cid}: {enhanced_cluster_details[cid]['name']}")
    time.sleep(60)  # stay under API rate limits

cluster_names = {cid: d["name"] for cid, d in enhanced_cluster_details.items()}

## 10. LLM: Enhanced Duplicate-Pair Insights

In [ ]:
def categorize_age(updated_date, current_date):
    if pd.isnull(updated_date):
        return "Unknown"
    age_days = (current_date - updated_date).days
    if age_days < 30:
        return "Active (< 30 days)"
    if age_days < 60:
        return "Recent (30-60 days)"
    if age_days < 180:
        return "Aging (> 60 days)"
    if age_days < 365:
        return "Stale (> 180 days)"
    return "Very Stale (> 365 days)"

duplicate_insight_prompt = PromptTemplate(
    input_variables=["release_notes_context", "project_context", "theme_2026_context",
                      "issue1_key", "issue1_summary", "issue1_description", "issue1_age_category",
                      "issue2_key", "issue2_summary", "issue2_description", "issue2_age_category", "similarity"],
    template=(
        "You are an expert in Jira issue analysis. Analyze the two issues below and assess whether "
        "they are duplicates, using the supplied context.\n\n"
        "Project context:\n{project_context}\n\n"
        "Release notes:\n{release_notes_context}\n\n"
        "2026 Themes:\n{theme_2026_context}\n\n"
        "Issue 1: {issue1_key} | {issue1_summary} | {issue1_description} | {issue1_age_category}\n"
        "Issue 2: {issue2_key} | {issue2_summary} | {issue2_description} | {issue2_age_category}\n"
        "Similarity Score: {similarity:.4f}\n\n"
        "Respond in JSON with keys `llm_reasoning` and `llm_recommendations`."
    )
)

current_date = pd.Timestamp.now(tz=timezone.utc)
enhanced_duplicate_insights = []

for i, pair in enumerate(duplicate_pairs):
    issue1 = df_jira_issues.loc[df_jira_issues["key"] == pair["issue1_key"]].iloc[0]
    issue2 = df_jira_issues.loc[df_jira_issues["key"] == pair["issue2_key"]].iloc[0]
    age1 = categorize_age(issue1["updated"], current_date)
    age2 = categorize_age(issue2["updated"], current_date)

    prompt = duplicate_insight_prompt.format(
        release_notes_context=release_notes_context,
        project_context=project_context,
        theme_2026_context=theme_2026_context,
        issue1_key=pair["issue1_key"], issue1_summary=pair["issue1_summary"],
        issue1_description=issue1["description"] or "No description provided.", issue1_age_category=age1,
        issue2_key=pair["issue2_key"], issue2_summary=pair["issue2_summary"],
        issue2_description=issue2["description"] or "No description provided.", issue2_age_category=age2,
        similarity=pair["similarity"],
    )
    try:
        result = call_llm_json(prompt)
    except json.JSONDecodeError:
        result = {"llm_reasoning": "Error processing LLM response.",
                  "llm_recommendations": "Error processing LLM response."}

    enhanced_duplicate_insights.append({
        "issue1_key": pair["issue1_key"], "issue1_age_category": age1,
        "issue2_key": pair["issue2_key"], "issue2_age_category": age2,
        "similarity": pair["similarity"],
        "llm_reasoning": result.get("llm_reasoning"),
        "llm_recommendations": result.get("llm_recommendations"),
    })
    print(f"Pair {i+1}: {pair['issue1_key']} & {pair['issue2_key']} - insight generated.")
    time.sleep(60)

## 11. LLM: Executive Summary

In [ ]:
executive_summary_prompt = PromptTemplate(
    input_variables=["project_context", "theme_2026_context", "enhanced_cluster_details", "enhanced_duplicate_insights"],
    template=(
        "You are an expert in Jira backlog analysis and strategic reporting. Write a concise executive "
        "summary (3-5 paragraphs) covering cluster analysis and duplicate detection findings, their "
        "alignment with the 2026 themes, and high-level PM recommendations.\n\n"
        "Project context:\n{project_context}\n\n"
        "2026 Themes:\n{theme_2026_context}\n\n"
        "Cluster analysis:\n{enhanced_cluster_details}\n\n"
        "Duplicate issue analysis:\n{enhanced_duplicate_insights}\n"
    )
)

clusters_summary = "\n".join(
    f"- Cluster {cid}: {d['name']}. Theme alignment: {d['theme_alignment']}. "
    f"Description: {d['description']}. PM recommendations: {d['pm_recommendations']}"
    for cid, d in enhanced_cluster_details.items()
)

duplicates_summary = "\n".join(
    f"- {ins['issue1_key']} ({ins['issue1_age_category']}) & {ins['issue2_key']} "
    f"({ins['issue2_age_category']}) sim={ins['similarity']:.4f}: {ins['llm_reasoning']} "
    f"Recommendation: {ins['llm_recommendations']}"
    for ins in enhanced_duplicate_insights
)

prompt = executive_summary_prompt.format(
    project_context=project_context,
    theme_2026_context=theme_2026_context,
    enhanced_cluster_details=clusters_summary,
    enhanced_duplicate_insights=duplicates_summary,
)

executive_summary = llm.invoke(prompt).content.strip()
with open("executive_summary.txt", "w") as f:
    f.write(executive_summary)

print(executive_summary)

## 12. Visualizations & Final Report

In [ ]:
# Issues per cluster
plt.figure(figsize=(10, 6))
cluster_counts.plot(kind="bar")
plt.xlabel("Cluster ID"); plt.ylabel("Number of Issues")
plt.title("Distribution of Issues Across Clusters")
plt.xticks(rotation=0); plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

# Issues per status
status_counts = df_jira_issues["status"].value_counts()
plt.figure(figsize=(12, 7))
sns.barplot(x=status_counts.index, y=status_counts.values, palette="viridis")
plt.xlabel("Issue Status"); plt.ylabel("Number of Issues")
plt.title("Distribution of Jira Issues by Status")
plt.xticks(rotation=45, ha="right"); plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout(); plt.show()

In [ ]:
print("Recommended issues for closing as duplicates:")
if enhanced_duplicate_insights:
    for ins in enhanced_duplicate_insights:
        print(f"- {ins['issue1_key']} & {ins['issue2_key']} (similarity: {ins['similarity']:.4f})")
        print(f"  Reasoning: {ins['llm_reasoning']}")
        print(f"  Recommendation: {ins['llm_recommendations']}\n")
else:
    print("No potential duplicate issues found above the similarity threshold.")